# UMelt: a high resolution melt map over Antactica

This notebook is a demonstrator from the **Digital Twin Component for Ice Sheets (DTC-IS)**, an ESA-funded, EO-driven digital twin project.

DTC-IS integrates Earth Observation, in-situ data, reanalysis products, and advanced data-driven models to provide a dynamic and science-based representation of the Greenland and Antarctic ice sheets, including their ice shelves. It supports the monitoring, understanding, and prediction of ice-sheet change, delivering actionable information for research, policy, and scenario-based exploration.

[Project Website](https://dtc-ice-sheets.org/) · [Interactive Dashboard](https://dashboards.dtc-ice-sheets.org/) · [Query API](https://query.dtc-ice-sheets.org/docs)

---

While the influence of surface melt on Antarctic ice shelf stability can be large, the duration and affected area
of melt events are often small. Therefore, melt events are difficult to capture with remote sensing, as satellite
sensors always face the trade-off between spatial and temporal resolution. To overcome this limitation, 
UMelt was developed: a surface melt record for all Antarctic ice shelves with a high spatial (500 m) and high
temporal (12 h) resolution for the period 2016–2021.

The model used to generate UMelt was created by Sophie de Rosa Husman et al. 
([link to the paper](https://doi.org/10.1016/j.rse.2023.113950)), based on the Attention UNet architecture.
Given that the input images retrieval and preprocessing needs considerable time and computing resources, only
precomputed results are shown within this DTC.
If you are interested to know more about pre-processing and model training you can visit the 
[github repository](https://github.com/SdeRodaHusman/UMelt/tree/main) linked to the paper.

---

**Key features:**
- **Location:** Draw a polygon on the map, to delineate the area of interest.
- **Analysis period:** Specify the start and end date within the UMelt availability window (2016-2023).
- **Animation:** Visualise the melt fraction evolving over time.

**How to use this notebook:**
1. Run the setup cells below.
2. **Select a polygon** on the map, around which a tile will be created.
3. Select a **time window**, for which to visualize the melt fraction.
4. Explore the melt maps over time, and their relation with the inputs

**Requirements:**
- Jupyter, VS Code with the Jupyter extension, or Google Colab.
- Internet access to the DTC-IS API.
- A DTC-IS API token (you will be asked for it below).

---

## Notebook preparations

In [ ]:
%matplotlib widget
import sys
from datetime import date

import plotly.io as pio
import xarray as xr
from IPython.display import display as ipydisplay

from dtc_is_notebook_helpers.authentication_widget import request_api_token
from dtc_is_notebook_helpers.cell_spinner import enable_cell_spinner
from dtc_is_notebook_helpers.uc4_umelt_plotting_helpers import (
    build_umelt_explorer,
    get_date_picker,
)

# Show an animation at the bottom of each cell while it is running
enable_cell_spinner()

if "google.colab" in sys.modules:
    pio.renderers.default = "colab"

### Enter DTC-IS API token

Please run the cell below, paste your DTC-IS API token into the box, and press 'Submit'. If you don’t have a token yet, you’ll need to generate one first using the link next to the box. A quick test query will check the token, and once it passes you can move on to the next cell! To submit a different token you will need to run the cell again.

---

In [ ]:
token_widget = request_api_token()
await token_widget.wait_until_authenticated()
client = token_widget.get_client()


## Select a location

Draw a polygon or a box on the map to select the area of interest. You can use the dropdown menu to select polygons previously drown.

In [ ]:
from src.dtc_is_notebook_helpers.uc4_umelt_plotting_helpers import polygon_selector

selector = polygon_selector()
display(selector)

## Select time window for the analysis

Choose the emulator model and the period to predict. ERA5 monthly means are available from 1940 to the present; the RACMO reference used for validation covers a shorter period. The values are read when you run the api below, so you can change them and re-run that cell without re-running this one.

In [ ]:
start_date_picker = get_date_picker(desc="Start date (from 2016 to 2021):", default_date=date(2018, 1, 1))
end_date_picker = get_date_picker(desc="End date (from 2016 to 2021):", default_date=date(2018, 6, 15))

ipydisplay(start_date_picker, end_date_picker)

## Call the API to retrieve the data

As explained above, the preprocessing required by the model is too complex to be run here, so precomputed results are offered.

Based on the time and area of interest selected, the API returns:
- **SSMIS**: Special Sensor Microwave Imager/Sounder observations
- **ASCAT**: Advanced Scatterometer data
- **UMelt**: Binary Melt Map obtained as UNet output (deep learning model)

In [ ]:
import asyncio

from src.dtc_is_notebook_helpers.uc4_umelt_plotting_helpers import run_data_download

polygon = selector.get_selected_polygon()

start_date, end_date = start_date_picker.value, end_date_picker.value
if start_date is None or end_date is None:
    raise ValueError("Please pick both a start and an end date above.")
if start_date >= end_date:
    raise ValueError("The start date must be before the end date.")

ssmis_res, ascat_res, umelt_res = await asyncio.gather(
    run_data_download(client, "SSMIS", start_date, end_date, polygon, ["brightness_temperature"]),
    run_data_download(client, "ASCAT", start_date, end_date, polygon,
                      ["surface_backwards_scattering_coefficient_of_radar_wave"]),
    run_data_download(client, "UC4_UMELT_PREDICTIONS", start_date, end_date, polygon, ["melt"]),
)

Open the downloaded datasets:

In [ ]:
ssmis_ds = xr.open_zarr(ssmis_res, consolidated=True)
ascat_ds = xr.open_zarr(ascat_res, consolidated=True)
umelt_ds = xr.open_zarr(umelt_res, consolidated=True)

## UMelt Explorer

This interactive interface helps you visualize the results.

On the first row it shows two of the model's inputs, SSMIS and ASCAT, and the models's outpu, UMelt.   
Below, the timeseries shows the average value at the selected location for each timestep within the selected window.

The model actually gets a third input, a climatology obtained from Sentinel-1 average backscatter intensity over 
the years.  
 Since Sentinel-1 data are not provided by the API they are not reported here. 


In [ ]:

fig_explorer = build_umelt_explorer(ssmis_ds, ascat_ds, umelt_ds)
fig_explorer.show()

## Visualize full Antarctica

> **WARNING**: Plotting data for full Antarctica requires a significant amount of memory.

Given the high resolution of UMelt, the size of the dataset covering whole Antarctica is big.  
For this reason, only one timestep can be selected for plotting.

In [ ]:
date_picker = get_date_picker(desc="Select a date (from 2016 to 2021):", default_date=date(2018, 1, 1))
ipydisplay(date_picker)

### Call the API

In [ ]:
from shapely.geometry import box

ANTARCTICA = box(-180, -90, 180, -60)

selected_date = date_picker.value
if selected_date is None:
    raise ValueError("Please pick both a start and an end date above.")

ssmis_res_ant, ascat_res_ant, umelt_res_ant = await asyncio.gather(
    run_data_download(client, "SSMIS", selected_date, None, ANTARCTICA, ["brightness_temperature"]),
    run_data_download(client, "ASCAT", selected_date, None, ANTARCTICA, ["surface_backwards_scattering_coefficient_of_radar_wave"]),
    run_data_download(client, "UC4_UMELT_PREDICTIONS", selected_date, None, ANTARCTICA, ["melt"]),
)

Open the downloaded datasets:

In [ ]:
ssmis_ant_ds = xr.open_zarr(ssmis_res_ant, consolidated=True).sel(time=start_date, method="nearest")
ascat_ant_ds = xr.open_zarr(ascat_res_ant, consolidated=True).sel(time=start_date, method="nearest")
umelt_ant_ds = xr.open_zarr(umelt_res_ant, consolidated=True).sel(time=start_date, method="nearest")

### Plot SSMIS, ASCAT and UMelt over Antactica

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

map_proj = ccrs.SouthPolarStereo(central_longitude=0.0)
fig, axes = plt.subplots(
    1, 3, figsize=(17, 11),
    subplot_kw={"projection": map_proj},
)

time_val = ssmis_ds.time.values
ssmis = ssmis_ds.brightness_temperature
ascat = ascat_ds.surface_backwards_scattering_coefficient_of_radar_wave
umelt = umelt_ds.melt

panels = [
    (ssmis, "SSMIS", "coolwarm", ssmis.min(), ssmis.max()),
    (ascat, "ASCAT", "viridis", ascat.min(), ascat.max()),
    (umelt, "UMelt", "cool", 0.0, 1.0)
]

for ax_i, (data, title, cmap, vmin, vmax) in zip(axes.flat, panels):
    data.plot(
        ax=ax_i, x="x", y="y",
        transform=ccrs.SouthPolarStereo(),
        cmap=cmap,
        vmin=vmin, vmax=vmax,
        cbar_kwargs={"shrink": 0.9, "orientation": "horizontal"},
    )
    ax_i.coastlines("10m")
    ax_i.gridlines(draw_labels=False)
    ax_i.set_title(title, fontsize=16)

plt.tight_layout()
plt.show()

## Conclusions

Thank you for exploring this DTC-IS demonstrator notebook. We hope it's given you a useful introduction to the creation of high resolution melt fields over Antarctica with a machine-learning trained on satellite observations, and to working with the DTC-IS framework — feel free to revisit it with different variables and parameters to explore further.

We welcome your feedback, questions, and suggestions for new features. You can reach the DTC-IS team at [support@dtc-ice-sheets.org](mailto:support@dtc-ice-sheets.org).